# 06 - NewsQA 200/11064 dataset EDA

Runs the analysis behind `docs/eda_report.md`. Each section is one script under
`scripts/eda/`; the scripts are the source of truth and this notebook is the
narrated driver, so a number here and a number in the report cannot drift apart.

Read-only over `data/evaluation/newsqa_200_11064/`. Nothing here writes to the
locked benchmark. Results are cached as JSON under `scripts/eda/out/`, so a
section can be re-read without re-running it.


In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / 'pyproject.toml').exists())
EDA = PROJECT_ROOT / 'scripts' / 'eda'
OUT = EDA / 'out'

def run(script: str) -> None:
    """Run one EDA script and stream its output."""
    process = subprocess.Popen(
        [sys.executable, str(EDA / script)],
        cwd=EDA, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding='utf-8', errors='replace', bufsize=1,
    )
    for line in process.stdout:
        print(line, end='')
    if process.wait() != 0:
        raise RuntimeError(f'{script} failed')

def cached(name: str) -> dict:
    """Read a previous result without re-running the script."""
    return json.loads((OUT / f'{name}.json').read_text(encoding='utf-8'))

print('project:', PROJECT_ROOT)
print('cached results:', len(list(OUT.glob('*.json'))))


## 1. Inventory, profile, cleanliness, integrity

The baseline picture: how many articles and questions, how long they are, how
many chunks they produce, what is mechanically dirty, and whether the ground
truth is internally consistent (evidence offsets resolve, gold chunk IDs exist,
variant ID sets agree).

Watch for two things. **1.74 chunks per article** means chunk retrieval is
almost article retrieval on this corpus, which caps how much a chunking
experiment can move. And the resolved questions are nearly twice as long as the
originals (11 words vs 6) - section 4 measures what that buys.

In [ ]:
run('01_profile.py')

## 2. Are the articles truncated?

Three independent checks, because the obvious argument does not hold. "The
maximum sits just under 4,600 characters, therefore truncation" is invalid -
there is no pile-up at the ceiling.

What does hold: the share of articles ending on sentence punctuation collapses
from 88.4% (shortest quartile) to 14.7% (longest quartile). Length-dependent
mid-sentence endings are truncation.

In [ ]:
run('01d_truncation.py')

### 2b. Proof by exact prefix

The archived CNN HTML under `data/cnn_downloads/` is the source the benchmark
was built from. Pairing benchmark articles against it: 50 of 94 pairs are an
exact character-for-character prefix of the archived page, at zero tolerance.
A prefix relationship is truncation by definition.

`01n_threshold_validity.py` then tests whether a length cut-off can stand in for
this check. It cannot: 40% of genuinely truncated articles are shorter than
3,800 characters.

In [ ]:
run('01e_truncation_proof.py')
run('01n_threshold_validity.py')

### 2c. Does truncation actually damage scoring?

Far less than the 41% headline suggests. NewsQA evidence is front-loaded - the
median answer sits at the 18th percentile of article length, and truncation
removes the tail. Only 9 questions have evidence both late in the article and in
the at-risk group.

In [ ]:
run('01m_truncation_impact.py')

## 3. Can the text be restored without re-crawling?

`data/cnn_downloads/cnn/downloads/` holds 24,469 archived CNN pages. This
section parses all of them (about a minute with 16 workers) and matches them to
the corpus by normalised prefix.

The answer to "did you fix the defect or re-crawl": the material is already in
the repository. 89.3% of the corpus matches its HTML, 1,927 articles have real
recoverable content, and only **33 of those are evaluation articles**.

Skip this cell if you only need the cached numbers - it is the one expensive
section.

In [ ]:
run('01p_full_coverage.py')   # ~1 minute, 16 processes
# cached('01p_full_coverage')  # or read the previous result instead

## 4. What was wrong with the questions, and did fixing it help?

The review recorded a reason code per question. Every code names a checkable
property of the question text - a missing subject, an unresolved pronoun - not a
judgement about difficulty. That is what makes resolution auditable instead of a
matter of taste.

The second table is the control: it measures how many *rare* terms each repair
added. Questions with no defect code gained zero words, which shows resolution
was not a blanket rewrite.

In [ ]:
run('05_reason_codes.py')

## 5. Question forms and answer types

Checks whether one question shape dominates enough that a single headline metric
describes that shape rather than the system.

In [ ]:
run('02_question_forms.py')

## 6. Lexical overlap - original vs resolved

Measures how much a question shares with its own gold chunk, against a
random-chunk baseline. This predicts sparse retrieval's advantage on this corpus
directly from the data, without appealing to a tournament result.

It is also the honest caveat: resolution raises rare-term anchors from 0.33 to
0.89 per question, and **a lexical retriever gains from that more than a dense
one does**. A sparse-vs-dense comparison run only on `resolved` is biased toward
sparse, which is why both variants have to be reported.

In [ ]:
run('03_lexical_overlap.py')

## 7. Competition and unlabelled answers

Two measurements that bound what any retrieval score can mean.

**Competition:** rare terms narrow 19,263 chunks to a median of 20 competitors,
and only 31% of questions narrow to 10 or fewer. First-stage retrieval gets
close and cannot finish - the dataset-level case for the reranker.

**Unlabelled answers:** how often a non-gold distractor contains the gold answer.
The raw string match (23.1%) overstates it, so matches are graded by how many of
the question's rare terms the distractor also carries. About 6.5% are strong
candidates - the same news event in a second article. That is the benchmark's
false-negative floor, and every reported Hit@K sits on top of it.

In [ ]:
run('04_distractor_collision.py')

## 8. Near-duplicate questions

The finding that decides the original-vs-resolved question.

In the **original** set, 34 pairs of near-identical questions point at different
articles - *"what does faa say"* appears three times with three different gold
articles. No retriever can separate those; they are unscoreable, not hard.
Resolution eliminates the class entirely (34 -> 0).

The cost appears in the same output: resolution collapsed 47 groups of
originally-distinct questions into word-for-word identical text, so 49 questions
are now scored twice on the same query. Effective distinct queries: 1,287.

In [ ]:
run('06_near_duplicates.py')

## 9. Cleaning

`scripts/clean_corpus.py` removes surviving page furniture - video teasers, the
share widget, publisher footers, block-boundary newline runs - and writes a
parallel `cleaned/` tree. The locked benchmark under `final/` is never touched.

Evidence spans are character offsets, so every deletion is recorded in an offset
map and each span is remapped and then re-verified against the cleaned text. A
span that cannot be resolved is flagged, never dropped.

This cell is a dry run. Add `--apply` to write.

In [ ]:
print(subprocess.run([sys.executable, 'scripts/clean_corpus.py'],
                     cwd=PROJECT_ROOT, capture_output=True, text=True,
                     encoding='utf-8', errors='replace').stdout)

## 10. Summary table

Pulls the headline numbers out of the cached results, so the report and this
notebook cannot disagree.

In [ ]:
rows = []
profile = cached('01_profile')
rows.append(('corpus articles', profile['inventory']['evaluation articles']
             + ' eval + ' + profile['inventory']['distractor articles'] + ' distractor'))
rows.append(('chunks @512/64', profile['inventory']['chunks @512/64']))
rows.append(('questions (resolved)', profile['inventory']['testset - resolved']))

truncation = cached('01d_truncation')
rows.append(('ends on a sentence, shortest quartile',
             f"{truncation['ends_cleanly']['shortest 25%']:.1%}"))
rows.append(('ends on a sentence, longest quartile',
             f"{truncation['ends_cleanly']['longest 25%']:.1%}"))

coverage = cached('01p_full_coverage')
rows.append(('articles matched to archived HTML',
             f"{coverage['matched']:,} of {coverage['corpus']:,}"))
rows.append(('evaluation articles with recoverable text',
             coverage['content_by_role'].get('evaluation', 0)))

overlap = cached('03_lexical_overlap')
rows.append(('rare anchors per question, original',
             f"{overlap['original']['rare_gold_mean']:.2f}"))
rows.append(('rare anchors per question, resolved',
             f"{overlap['resolved']['rare_gold_mean']:.2f}"))

collision = cached('04_distractor_collision')
rows.append(('median non-gold competitors', collision['competitors_median']))
rows.append(('unlabelled answers, strong / upper bound',
             f"{collision['strong_candidates']/collision['checked']:.1%}"
             f" / {collision['distractor_has_answer']/collision['checked']:.1%}"))

before = cached('06_near_duplicates_original')
after = cached('06_near_duplicates_resolved')
rows.append(('unscoreable cross-article near-duplicates',
             f"{before['cross_article_conflicts']} -> {after['cross_article_conflicts']}"))
rows.append(('duplicate resolved queries', cached('06_convergence')['surplus']))

width = max(len(str(k)) for k, _ in rows)
for key, value in rows:
    print(f'{key:<{width}}  {value}')

## 11. Figures

Renders the five figures embedded in `docs/eda_report.md` from the cached
results, so a number in the report and a number in a chart cannot drift apart.
300 DPI PNGs land in `docs/figures/eda/`.

The colours are categorical slots 1-3 of a CVD-validated palette, assigned by
entity (original vs resolved) rather than by rank, with direct value labels on
every bar.

In [ ]:
run('07_figures.py')

from IPython.display import Image, display
for name in ('fig1_truncation', 'fig2_evidence_position', 'fig3_question_repair',
             'fig4_retrieval_difficulty', 'fig5_restoration'):
    display(Image(filename=str(PROJECT_ROOT / 'docs' / 'figures' / 'eda' / f'{name}.png')))